# F-003-3: YOLOv8n 人物検出モデル学習 Notebook (PyTorch/Ultralytics版)

転倒検出用の人物検出モデルを Google Colab 上で学習し、
ONNX / TFLite INT8 形式にエクスポートする。

## 前提条件
- Google Colab (GPU ランタイム: T4 推奨)
- Google Drive に `fall_detection_dataset.zip` をアップロード済み

## ワークフロー
1. GPU確認・環境構築
2. データセット準備
3. モデル学習
4. 精度評価 (mAP)
5. ONNX エクスポート
6. TFLite FP32/INT8 変換
7. 成果物ダウンロード

---
## Step 1: GPU確認・環境構築

**重要:** メニューの「ランタイム > ランタイムのタイプを変更」で **GPU (T4)** を選択してください。

In [ ]:
# GPU 確認
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('=== GPU が利用可能です ===')
else:
    print('WARNING: GPU が検出されません。ランタイムを GPU に変更してください。')

In [ ]:
# Ultralytics YOLOv8 と変換ツールのインストール
!pip install -q ultralytics onnx onnx2tf onnxsim
print('=== インストール完了 ===')

---
## Step 2: データセット準備

### 事前準備 (ローカルPCで実行)

```bash
cd mimamori-sense/dataset/merged
zip -r fall_detection_dataset.zip images/ labels/
```

作成した `fall_detection_dataset.zip` を Google Drive のマイドライブ直下にアップロードしてください。

### データセット構成
```
images/
  train/  ← 学習用画像
  val/    ← 検証用画像
  test/   ← テスト用画像
labels/
  train/  ← 学習用ラベル (YOLO形式 .txt)
  val/    ← 検証用ラベル
  test/   ← テスト用ラベル
```

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

WORK_DIR = '/content/yolo_train'
DATASET_DIR = os.path.join(WORK_DIR, 'dataset')
DATASET_ZIP = '/content/drive/MyDrive/fall_detection_dataset.zip'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'作業ディレクトリ: {WORK_DIR}')

# データセット展開
if not os.path.isdir(DATASET_DIR):
    if os.path.isfile(DATASET_ZIP):
        print('データセット展開中...')
        !mkdir -p {DATASET_DIR} && unzip -q {DATASET_ZIP} -d {DATASET_DIR}
        print('展開完了')
    else:
        print(f'ERROR: {DATASET_ZIP} が見つかりません')
        print('Google Drive にアップロードしてください')
else:
    print('データセットは展開済みです')

# 検証
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if os.path.isdir(img_dir):
        img_count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        lbl_count = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.isdir(lbl_dir) else 0
        print(f'  {split}: images={img_count}, labels={lbl_count}')
    else:
        print(f'  WARNING: {img_dir} が見つかりません')

In [ ]:
# data.yaml 作成 (Ultralytics形式)
data_yaml = f"""path: {DATASET_DIR}
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: person
"""

data_yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml)

print(f'data.yaml 作成完了: {data_yaml_path}')
print()
print(data_yaml)

---
## Step 3: モデル学習

YOLOv8n (nano) モデルを使用。Ultralytics の学習パイプラインが
データ読み込み・前処理・学習・評価をすべて自動で処理する。

- **モデル**: YOLOv8n (最小構成、約3.2Mパラメータ)
- **入力サイズ**: 192x192
- **クラス数**: 1 (person)
- **エポック数**: 100 (T4 GPU で約1-2時間)

**注意:** 学習中に Colab のセッションが切れないよう注意してください。

In [ ]:
from ultralytics import YOLO

# YOLOv8n (nano) をベースに学習
model = YOLO('yolov8n.pt')  # COCO事前学習済み重みをロード

results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=192,
    batch=64,
    device=0,           # GPU
    workers=2,
    project=WORK_DIR,
    name='train',
    exist_ok=True,
    # 学習パラメータ
    optimizer='SGD',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    # データ拡張
    hsv_h=0.0,          # グレースケール運用を想定し色相変換を無効化
    hsv_s=0.0,          # 彩度変換も無効化
    hsv_v=0.4,          # 明度変換のみ有効
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)

print('\n=== 学習完了 ===')

In [ ]:
# 学習曲線の表示
from IPython.display import Image, display
import os

results_png = os.path.join(WORK_DIR, 'train', 'results.png')
if os.path.exists(results_png):
    display(Image(filename=results_png, width=800))
else:
    print('学習結果の画像が見つかりません')

---
## Step 4: 精度評価 (mAP)

検証データセットでモデルの精度を評価する。

- **mAP@0.5**: IoU閾値0.5での平均精度（目標: 90%以上）
- **mAP@0.5:0.95**: IoU閾値0.5~0.95での平均精度

In [ ]:
# best.pt で検証データセットを評価
best_pt = os.path.join(WORK_DIR, 'train', 'weights', 'best.pt')
model = YOLO(best_pt)

metrics = model.val(
    data=data_yaml_path,
    imgsz=192,
    batch=64,
    device=0,
    split='val',
)

print(f'\n=== 評価結果 ===')
print(f'mAP@0.5     : {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

# KPI 判定
if metrics.box.map50 >= 0.9:
    print('\n>>> KPI達成: mAP@0.5 >= 90%')
else:
    print(f'\n>>> KPI未達: mAP@0.5 = {metrics.box.map50*100:.1f}% (目標: 90%以上)')
    print('    エポック数を増やすか、データセットの見直しを検討してください')

In [ ]:
# テストデータセットでも評価
test_metrics = model.val(
    data=data_yaml_path,
    imgsz=192,
    batch=64,
    device=0,
    split='test',
)

print(f'\n=== テストデータ評価結果 ===')
print(f'mAP@0.5     : {test_metrics.box.map50:.4f} ({test_metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {test_metrics.box.map:.4f} ({test_metrics.box.map*100:.1f}%)')
print(f'Precision    : {test_metrics.box.mp:.4f}')
print(f'Recall       : {test_metrics.box.mr:.4f}')

---
## Step 5: ONNX エクスポート

In [ ]:
# ONNX エクスポート
model = YOLO(best_pt)

onnx_path = model.export(
    format='onnx',
    imgsz=192,
    opset=11,
    simplify=True,
)

print(f'\nONNX エクスポート完了: {onnx_path}')
print(f'サイズ: {os.path.getsize(onnx_path)/1024:.1f} KB')

---
## Step 6: TFLite FP32/INT8 変換

In [ ]:
import numpy as np
import glob
from PIL import Image

ONNX_PATH = onnx_path
SAVED_MODEL_DIR = os.path.join(WORK_DIR, 'saved_model')
FP32_PATH = os.path.join(WORK_DIR, 'model_fp32.tflite')
INT8_PATH = os.path.join(WORK_DIR, 'model_int8.tflite')
IMG_SIZE = 192

# Step 6a: ONNX → SavedModel → TFLite FP32
print('=== ONNX → SavedModel (onnx2tf) ===')
!onnx2tf -i {ONNX_PATH} -o {SAVED_MODEL_DIR} -osd 2>&1 | tail -10

if os.path.isdir(SAVED_MODEL_DIR):
    import tensorflow as tf

    # FP32 TFLite
    print('\n=== FP32 TFLite 変換 ===')
    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    tflite_fp32 = converter.convert()
    with open(FP32_PATH, 'wb') as f:
        f.write(tflite_fp32)
    print(f'FP32 TFLite: {os.path.getsize(FP32_PATH)/1024:.1f} KB')

    # Step 6b: INT8 量子化
    print('\n=== INT8 量子化 ===')
    cal_dir = os.path.join(DATASET_DIR, 'images', 'val')
    cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.jpg')))[:200]
    if not cal_images:
        cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.png')))[:200]
    print(f'キャリブレーション画像: {len(cal_images)}枚')

    # FP32モデルの入力形状を取得
    interp = tf.lite.Interpreter(model_path=FP32_PATH)
    interp.allocate_tensors()
    inp_detail = interp.get_input_details()[0]
    inp_shape = inp_detail['shape']
    n, h, w, c = inp_shape
    print(f'入力形状: {inp_shape} (NHWC)')

    def representative_dataset():
        for img_path in cal_images:
            img = Image.open(img_path).convert('RGB').resize((w, h))
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = arr.reshape(inp_shape)
            yield [arr]

    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    try:
        int8_model = converter.convert()
        with open(INT8_PATH, 'wb') as f:
            f.write(int8_model)
        int8_kb = os.path.getsize(INT8_PATH) / 1024
        print(f'INT8 TFLite: {int8_kb:.1f} KB')
    except Exception as e:
        print(f'INT8 量子化エラー: {e}')
        print('FP32 モデルは正常に生成されています。')
else:
    print('ERROR: SavedModel の生成に失敗しました')

In [ ]:
# モデル検査
import tensorflow as tf
import numpy as np

print('=== モデルサイズ比較 ===')
for label, path in [('FP32', FP32_PATH), ('INT8', INT8_PATH)]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'{label}: {size_kb:.1f} KB ({size_kb/1024:.2f} MB)')

if os.path.exists(INT8_PATH):
    print('\n=== INT8 モデル詳細 ===')
    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    for tag, details in [('Input', interp.get_input_details()),
                         ('Output', interp.get_output_details())]:
        print(f'\n--- {tag} ---')
        for i, d in enumerate(details):
            print(f'  [{i}] {d["name"]} shape={d["shape"]} dtype={d["dtype"]}')
            qp = d.get('quantization_parameters', {})
            sc = qp.get('scales', np.array([]))
            zp = qp.get('zero_points', np.array([]))
            if len(sc) > 0:
                print(f'      scale={sc[0]:.8f}, zero_point={zp[0]}')

---
## Step 7: 成果物ダウンロード

学習済みモデルを Google Drive に保存する。

In [ ]:
# Google Drive に成果物をコピー
import shutil

OUTPUT_DIR = '/content/drive/MyDrive/fall_detection_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# コピー対象ファイル
files_to_copy = {
    os.path.join(WORK_DIR, 'train', 'weights', 'best.pt'): 'best.pt',
    os.path.join(WORK_DIR, 'train', 'weights', 'last.pt'): 'last.pt',
    os.path.join(WORK_DIR, 'train', 'results.png'): 'results.png',
    os.path.join(WORK_DIR, 'train', 'results.csv'): 'results.csv',
    ONNX_PATH: 'model.onnx',
    FP32_PATH: 'model_fp32.tflite',
    INT8_PATH: 'model_int8.tflite',
}

for src, dst_name in files_to_copy.items():
    if os.path.exists(src):
        dst = os.path.join(OUTPUT_DIR, dst_name)
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(src) / 1024
        print(f'  {dst_name}: {size_kb:.1f} KB')

print(f'\n=== Google Drive に保存完了 ===')
print(f'場所: {OUTPUT_DIR}')

In [ ]:
# INT8 TFLite モデルを直接ダウンロード
from google.colab import files

if os.path.exists(INT8_PATH):
    files.download(INT8_PATH)
    print('INT8モデルのダウンロードを開始しました')
else:
    print('INT8モデルが見つかりません。Step 6 を先に実行してください。')

---
## まとめ

### 生成される成果物

| ファイル | 説明 |
|---|---|
| `best.pt` | 学習済み PyTorch モデル (最良 mAP) |
| `last.pt` | 最終エポックのモデル |
| `model.onnx` | ONNX 形式モデル |
| `model_fp32.tflite` | TFLite FP32 モデル |
| `model_int8.tflite` | TFLite INT8 量子化モデル |
| `results.png` | 学習曲線チャート |
| `results.csv` | 学習ログ (CSV) |

### 次のステップ

1. INT8 モデルのサイズが MCU のメモリ制約に収まることを確認
2. RUHMI/MERA SDK で Ethos-U55 向けに最適化 (必要に応じて)
3. 生成コードを `e2studio_CPU0/src/ai_application/` に配置
4. 実機 (EK-RA8P1) での動作確認

### 備考

- 本ノートブックは RGB (3チャネル) で学習しています
- MCU でグレースケール (1チャネル) カメラを使用する場合、
  デプロイ時にグレースケール→RGB変換 (チャネル複製) を行うか、
  1チャネル入力モデルに再学習する必要があります
- モデルサイズが MCU の制約を超える場合、YOLOv8n より小さい
  カスタムモデルの検討が必要です